# `run_eval_pipeline.ipynb` — speculative-decoding eval driver (RunAI)

Notebook equivalent of `scripts/run_eval_pipeline.py`. It drives the two-phase
eval contract documented in `CLAUDE.md` (Eval contract — `scripts/evaluate_sd.py`):

1. `engine=hf` — HF target+draft, instrumented loop in `src/kdsd/sd/instrument.py`
   produces `acceptance_rate`, `accepted_lens`, generations, an HF-side speedup,
   and writes `results/<run_name>/eval_summary.json`.
2. `engine=vllm` — vanilla and SD passes in vLLM spawn subprocesses; merges a
   `vllm` block into `eval_summary.json` and overwrites the top-level
   `sd_time_s`/`vanilla_time_s`/`speedup`/`tokens_per_second` when SD succeeds.

Each phase runs as a **separate `subprocess.run`** so vLLM never shares a CUDA
context with HF — vLLM cannot release its CUDA context cleanly, so the second
engine load OOMs in the same process. This matches `scripts/run_eval_pipeline.py`.

## How to run

1. From your laptop, submit the interactive Jupyter pod with `notebooks/submit.sh`
   (set `GASPAR` and `GROUP` first, see `rcp_support/README.md`).
2. `runai port-forward <job-name> --port 8888:8888` — open
   `http://localhost:8888` (token `cs552`).
3. Inside Jupyter, clone this repo under `/scratch/` and open this notebook from
   `/scratch/<repo>/notebooks/run_eval_pipeline.ipynb`.
4. Edit the **Configuration** cell, then `Run All`.

**vLLM CUDA caveat** (per `rcp_support/README.md` §vLLM and CUDA): in this
notebook we never `import vllm` or call `torch.cuda.*` in the parent kernel; both
phases run inside child processes. The kernel itself can be restarted between
runs without losing state.

## 1. Environment

`HF_HOME=/scratch/hf_cache` is already set by `notebooks/submit.sh`. We re-assert
it here so the notebook also runs from a `runai bash` shell or a non-default
launcher. `evaluate_sd.py` reads `cfg.hf_cache` and re-exports `HF_HOME` /
`HF_HUB_CACHE` / `HF_DATASETS_CACHE` before importing transformers.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)
print("repo root:", REPO)

os.environ.setdefault("HF_HOME", "/scratch/hf_cache")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Bootstrap the `kdsd` package

The course image (`registry.rcp.epfl.ch/course-cs-552/base-vllm:v1`) already
ships torch 2.8 / transformers 4.57 / vLLM 0.11 / hydra-core. We only need the
local `kdsd` package on the import path.

In [ ]:
!pip install --quiet -e .

## 3. Configuration

Edit these for your run. `hydra_overrides` is forwarded verbatim to both phases
(same grammar as `uv run python scripts/evaluate_sd.py engine=... <overrides>`).

In [ ]:
RUN_NAME = "spec_smoke"
DRAFT = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPTS_JSONL = "data/processed/eval.jsonl"
PROMPTS_LIMIT = 20
GAMMA = 4
MAX_NEW_TOKENS = 256

hydra_overrides = [
    f"run_name={RUN_NAME}",
    f"draft={DRAFT}",
    f"prompts.jsonl={PROMPTS_JSONL}",
    f"prompts.limit={PROMPTS_LIMIT}",
    f"runtime.gamma={GAMMA}",
    f"runtime.max_new_tokens={MAX_NEW_TOKENS}",
]

RESULTS_DIR = REPO / "results" / RUN_NAME
print("hydra overrides:", hydra_overrides)
print("results dir:", RESULTS_DIR)

In [ ]:
import subprocess

EVAL_SCRIPT = REPO / "scripts" / "evaluate_sd.py"

def run_phase(engine: str, overrides: list[str]) -> int:
    cmd = [sys.executable, str(EVAL_SCRIPT), f"engine={engine}", *overrides]
    print(f"\n>>> {' '.join(cmd)}\n", flush=True)
    proc = subprocess.Popen(
        cmd, cwd=str(REPO),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"\n<<< {engine} phase exit code: {rc}", flush=True)
    return rc

## 4. Phase 1 — HF instrumented loop

Loads the HF target and (when `draft` is set) the HF draft, runs the
instrumented SD loop in `src/kdsd/sd/instrument.py`, writes
`results/<run_name>/{eval_summary.json,generations.jsonl,timing.json,config.yaml}`.
This is the sole source of trusted per-step metrics (`acceptance_rate`,
`avg_accepted_tokens`, per-prompt `accepted_lens`).

In [ ]:
rc = run_phase("hf", hydra_overrides)
if rc != 0:
    raise RuntimeError(f"HF phase failed (rc={rc}); skipping vLLM phase")

## 5. Phase 2 — vLLM vanilla + SD

Runs in a fresh subprocess (vLLM cannot share its CUDA context with HF — see
`scripts/run_eval_pipeline.py` and `src/kdsd/sd/vllm_runner.py`). Reads the
`eval_summary.json` from Phase 1, runs vanilla and SD passes through vLLM in
spawn subprocesses, merges a `vllm` block into the same summary, and overwrites
the top-level `sd_time_s`/`vanilla_time_s`/`speedup`/`tokens_per_second` when
SD succeeds.

In [ ]:
rc = run_phase("vllm", hydra_overrides)
if rc != 0:
    print(f"WARNING: vLLM phase failed (rc={rc}); HF results are still on disk.")

## 6. Inspect the merged summary

The schema is frozen in `CLAUDE.md` (Eval contract). `engines.{hf,vllm}`
preserves both engines' raw numbers regardless of which overwrites the top-level
fields.

In [ ]:
import json

summary_path = RESULTS_DIR / "eval_summary.json"
summary = json.loads(summary_path.read_text())

top_keys = (
    "acceptance_rate", "avg_accepted_tokens",
    "vanilla_time_s", "sd_time_s", "speedup", "tokens_per_second",
    "n_prompts", "n_warmup", "n_repeats",
)
for k in top_keys:
    print(f"{k:25s} {summary.get(k)}")

print("\nengines:")
print(json.dumps(summary.get("engines", {}), indent=2))

In [ ]:
gens_path = RESULTS_DIR / "generations.jsonl"
with gens_path.open() as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        rec = json.loads(line)
        prompt = rec.get("prompt", "")
        gen = rec.get("generation", "")
        print(f"--- record {i} ---")
        print("prompt   :", prompt[:200].replace("\n", " "))
        print("generation:", gen[:200].replace("\n", " "))
        print("accepted_lens:", rec.get("accepted_lens"))
        print()